# RAG Document Q&A System
## Quarterly Earnings Reports — Local Pipeline Walkthrough

This notebook walks through every stage of the RAG pipeline:

| # | Section | What it does |
|---|---------|-------------|
| 1 | **Setup & Imports** | Install deps, load env, verify config |
| 2 | **Load & Chunk PDFs** | Ingest reports, inspect chunks & metadata |
| 3 | **Build Vector Store** | Embed chunks, persist FAISS index |
| 4 | **Sample Queries** | 5 natural language questions |
| 5 | **Metadata Filtering** | Query a specific company / quarter |
| 6 | **Hallucination Check** | RAG answer vs. vanilla LLM |
| 7 | **Evaluation** | Retrieval relevance for 3 test questions |

> **Before you begin:** Add PDF files to `data/reports/` and set `OPENAI_API_KEY` in `.env`.

---
## Section 1 — Setup & Imports

In [ ]:
# Install dependencies (run once; comment out after first execution)
# !pip install -r requirements.txt

In [1]:
import sys
import os
from pathlib import Path

# ── Ensure the repo root is on the path so src/ modules resolve correctly ──
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

# ── Make the project virtualenv site-packages available even when the notebook
#    is launched with a different interpreter/kernel. ───────────────────────
VENV_SITE_PACKAGES = sorted((REPO_ROOT / ".venv" / "lib").glob("python*/site-packages"))
for site_packages in VENV_SITE_PACKAGES:
    site_packages_str = str(site_packages)
    if site_packages_str not in sys.path:
        sys.path.insert(0, site_packages_str)

# ── Load .env (OPENAI_API_KEY etc.) ──────────────────────────────────────
from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

# ── Project imports ───────────────────────────────────────────────────────
import config
from pdf_loader import load_and_chunk_pdfs
from embedder  import build_vectorstore, load_vectorstore, get_embedding_model
from retriever import similarity_search, format_sources
from qa_chain  import build_qa_chain, ask, ask_vanilla_llm, pretty_print

print("✓ Imports OK")
print(f"  DATA_DIR       : {config.DATA_DIR}")
print(f"  VECTORSTORE_DIR: {config.VECTORSTORE_DIR}")
print(f"  LLM model      : {config.LLM_MODEL}")
print(f"  Embedding model: {config.HF_EMBEDDING_MODEL}")
print(f"  OPENAI_API_KEY : {'set ✓' if os.environ.get('OPENAI_API_KEY') else 'NOT SET ✗'}")

✓ Imports OK
  DATA_DIR       : /Users/angelomiranda/Desktop/rag-doc-qa/data/reports
  VECTORSTORE_DIR: /Users/angelomiranda/Desktop/rag-doc-qa/vectorstore
  LLM model      : gpt-4o
  Embedding model: all-MiniLM-L6-v2
  OPENAI_API_KEY : set ✓


---
## Section 2 — Load & Chunk PDFs

Scans `data/reports/` for `.pdf` files, extracts text with PyMuPDF,
attaches metadata (source, company, quarter, year, page), and splits
into overlapping chunks.

In [2]:
# ── Load & chunk all PDFs ─────────────────────────────────────────────────
chunks = load_and_chunk_pdfs(config.DATA_DIR)

print(f"\nTotal chunks : {len(chunks)}")

  Loading: APPL_Q2_2026.pdf
  Loading: LITE_Q3_2026.pdf
  Loading: MSFT_Q3_2026.pdf
  Loading: MU_Q3_2026.pdf
  Loading: NBIS_Q1_2026.pdf
  Loading: TER_Q1_2026.pdf
  Loading: TSLA_FY_2025.pdf

[INFO] Loaded 260 pages from 7 PDF(s).
[INFO] Split 260 pages → 1653 chunks (size=500, overlap=50).

Total chunks : 1653


In [3]:
# ── Inspect a sample chunk ────────────────────────────────────────────────
if chunks:
    sample = chunks[0]
    print("── Sample chunk ──────────────────────────────────────────────")
    print(sample.page_content[:500])
    print("\n── Metadata ──────────────────────────────────────────────────")
    for k, v in sample.metadata.items():
        print(f"  {k:10}: {v}")
else:
    print("No chunks yet — add PDFs to data/reports/ and re-run.")

── Sample chunk ──────────────────────────────────────────────
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-Q
(Mark One)
☒ QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the quarterly period ended March 28, 2026
or
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from              to             .
Commission File Number: 001-36743
Apple Inc.
(Exact name of Registrant as specified in its charter)
California

── Metadata ──────────────────────────────────────────────────
  source    : APPL_Q2_2026.pdf
  company   : APPL
  quarter   : Q2
  year      : 2026
  page      : 1


In [4]:
# ── Chunk distribution by company ─────────────────────────────────────────
import pandas as pd

if chunks:
    df = pd.DataFrame([c.metadata for c in chunks])
    print(df.groupby(["company", "quarter", "year"]).size()
            .rename("chunk_count")
            .reset_index()
            .to_string(index=False))

company quarter year  chunk_count
   APPL      Q2 2026          228
   LITE      Q3 2026           65
   MSFT      Q3 2026          794
     MU      Q3 2026           89
   NBIS      Q1 2026           57
    TER      Q1 2026           37
   TSLA      FY 2025          383


---
## Section 3 — Build Vector Store

Embeds all chunks with **HuggingFace all-MiniLM-L6-v2** (runs offline)
and persists a FAISS index to `vectorstore/`.  
Re-running this cell loads from cache — no re-embedding needed.

In [5]:
# ── Build (or load) FAISS vectorstore ────────────────────────────────────
vectorstore = build_vectorstore(chunks, persist_dir=config.VECTORSTORE_DIR)

print(f"\nEmbedding dimensions : {vectorstore.index.d}")
print(f"Vectors in index     : {vectorstore.index.ntotal}")

[INFO] Cached vectorstore found at '/Users/angelomiranda/Desktop/rag-doc-qa/vectorstore'. Loading…
[INFO] Embedding model: HuggingFace / all-MiniLM-L6-v2


/Users/angelomiranda/Desktop/rag-doc-qa/src/embedder.py:64: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(model_name=HF_EMBEDDING_MODEL)


[INFO] Vectorstore loaded from '/Users/angelomiranda/Desktop/rag-doc-qa/vectorstore'.

Embedding dimensions : 384
Vectors in index     : 1653


In [6]:
# ── Quick sanity-check retrieval ──────────────────────────────────────────
test_results = vectorstore.similarity_search("revenue growth", k=3)
print(f"Sanity-check — top 3 chunks for 'revenue growth':")
for i, doc in enumerate(test_results, 1):
    src = doc.metadata.get("source", "?")
    pg  = doc.metadata.get("page", "?")
    print(f"  [{i}] {src}  p.{pg}  →  {doc.page_content[:120].strip()!r}")

Sanity-check — top 3 chunks for 'revenue growth':
  [1] MSFT_Q3_2026.pdf  p.28  →  'Revenue\n $\n13,192  $\n13,371  $\n41,198  $\n41,198 \nCost of revenue\n  \n5,511   \n6,095   \n17,821   \n19,111 \nOperating expens'
  [2] MSFT_Q3_2026.pdf  p.51  →  'competitors.\nIt is uncertain whether our strategies will continue to attract users or generate the revenue required to s'
  [3] MU_Q3_2026.pdf  p.33  →  'June 24, 2026\n33\n33\nAmounts in millions\nFQ3-26\n% of \nrevenue\nFQ2-26\n% of \nrevenue\nFQ3-25\n% of \nrevenue\nDRAM\n$31,328\n76%'


---
## Section 4 — Run Sample Queries

Five representative earnings-report questions routed through the full RAG chain.

In [7]:
# ── Build the QA chain (built once, reused below) ─────────────────────────
qa_chain = build_qa_chain(k=config.TOP_K)
print("QA chain ready.")

[INFO] Embedding model: HuggingFace / all-MiniLM-L6-v2
[INFO] Vectorstore loaded from '/Users/angelomiranda/Desktop/rag-doc-qa/vectorstore'.
QA chain ready.


In [20]:
# ── Query 1: Apple revenue ────────────────────────────────────────────────
q1 = "Did iPhone and Mac sales increase?"
r1 = ask(q1, qa_chain=qa_chain)
print(f"Q: {q1}")
pretty_print(r1)

Q: Did iPhone and Mac sales increase?
ANSWER
Yes, iPhone sales increased during both the second quarter and the first six months of 2026 compared to the same periods in 2025. Specifically, iPhone net sales were $56,994 million in the second quarter of 2026 compared to $46,841 million in the second quarter of 2025, representing a 22% increase. For the first six months, iPhone net sales were $142,263 million in 2026 compared to $115,979 million in 2025, representing a 23% increase (Apple Inc. | Q2 2026 Form 10-Q | Page 15).

Mac sales increased during the second quarter of 2026 compared to the second quarter of 2025, with net sales of $8,399 million in 2026 compared to $7,949 million in 2025, representing a 6% increase. However, year-over-year Mac net sales during the first six months of 2026 were relatively flat, with $16,785 million in 2026 compared to $16,936 million in 2025, representing a 1% decrease (Apple Inc. | Q2 2026 Form 10-Q | Page 15).

SOURCES
------------------------------

In [21]:
# ── Query 2: Microsoft cloud revenue ─────────────────────────────────────
q2 = "What was Microsoft's cloud revenue increase? How much in billions"
r2 = ask(q2, qa_chain=qa_chain)
print(f"Q: {q2}")
pretty_print(r2)

Q: What was Microsoft's cloud revenue increase? How much in billions
ANSWER
Microsoft's cloud revenue increased by $54.5 billion. (Source: Highlights from the third quarter of fiscal year 2026 compared with the third quarter of fiscal year 2025)

SOURCES
----------------------------------------------------------------------
  • MSFT_Q3_2026.pdf (p.35)
  • MSFT_Q3_2026.pdf (p.31)
  • MSFT_Q3_2026.pdf (p.39)
  • MSFT_Q3_2026.pdf (p.37)
